# LeetCode #148 — Sort List: merge sort trên danh sách liên kết

**Mức độ:** Medium · **Chủ đề:** Linked List, Two Pointers, Divide and Conquer, Sorting, Merge Sort

Skeleton trên LeetCode:

```python
class Solution:
    def sortList(self, head: Optional[ListNode]) -> Optional[ListNode]:
```

Mục tiêu của notebook:

1. Vì sao **merge sort** hợp với danh sách liên kết (trộn không cần mảng phụ; không cần truy cập ngẫu nhiên).
2. **Top-down:** tìm giữa (nhanh/chậm) → cắt → đệ quy → trộn. `O(log n)` ngăn xếp đệ quy.
3. **Bottom-up** (follow-up `O(1)` bộ nhớ): trộn các đoạn độ dài `1, 2, 4, …` bằng vòng lặp.
4. So sánh với insertion sort (#147) và quick sort trên danh sách.

# 1. Bài toán là gì?

Cho `head` của danh sách liên kết, trả về danh sách đã **sắp xếp tăng dần**.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 5 * 10^4]`.
- `-10^5 <= Node.val <= 10^5`

**Follow up (theo đề):** sắp xếp trong `O(n log n)` thời gian và `O(1)` bộ nhớ (hằng số)?

---

## Ví dụ

```text
4 → 2 → 1 → 3            ->  1 → 2 → 3 → 4
-1 → 5 → 3 → 4 → 0       ->  -1 → 0 → 3 → 4 → 5
[]                       ->  []
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)
2. Sắp xếp (sorting)
3. Chia để trị (divide and conquer)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [6]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [7]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [8]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.3 Chia để trị (divide and conquer)

### Ý tưởng

1. **Chia** bài toán thành các bài con **cùng dạng**, nhỏ hơn (thường là hai nửa).
2. **Trị**: giải các bài con bằng đệ quy (bài đủ nhỏ thì giải trực tiếp).
3. **Kết hợp** kết quả các bài con thành kết quả bài lớn.

Khác DP: các bài con **không chồng lấp** (mỗi phần dữ liệu thuộc đúng một bài con).

```text
            [8 phần tử]                      tầng 0: 1 bài, tổng n việc
         [4]          [4]                    tầng 1: 2 bài, tổng n việc
       [2]  [2]     [2]  [2]                 tầng 2: 4 bài, tổng n việc
      [1][1][1][1] [1][1][1][1]              log₂ n tầng  →  O(n log n)
```

In [9]:
def max_chia_de_tri(a, lo, hi, sau=0):
    """Phần tử lớn nhất của a[lo..hi]."""
    if lo == hi:
        return a[lo]                                   # bài đủ nhỏ
    mid = (lo + hi) // 2
    trai = max_chia_de_tri(a, lo, mid, sau + 1)        # trị nửa trái
    phai = max_chia_de_tri(a, mid + 1, hi, sau + 1)    # trị nửa phải
    kq = max(trai, phai)                               # kết hợp
    print("    " * sau + f"a[{lo}..{hi}] = {a[lo:hi + 1]} → max {kq}")
    return kq

_ = max_chia_de_tri([3, 8, 1, 9, 4, 7, 2, 6], 0, 7)

        a[0..1] = [3, 8] → max 8
        a[2..3] = [1, 9] → max 9
    a[0..3] = [3, 8, 1, 9] → max 9
        a[4..5] = [4, 7] → max 7
        a[6..7] = [2, 6] → max 6
    a[4..7] = [4, 7, 2, 6] → max 7
a[0..7] = [3, 8, 1, 9, 4, 7, 2, 6] → max 9


### Các ví dụ quen thuộc

| Bài | Chia | Kết hợp | Độ phức tạp |
|---|---|---|---|
| Tìm nhị phân | bỏ một nửa | không cần | `O(log n)` |
| Merge sort | hai nửa | trộn `O(n)` | `O(n log n)` |
| Lũy thừa nhanh `x^n` | `x^(n/2)` | bình phương | `O(log n)` |
| Dựng cây cân bằng từ mảng sắp xếp | phần tử giữa làm gốc | gắn hai cây con | `O(n)` |
| Dựng cây từ hai cách duyệt | tách theo vị trí gốc | gắn hai cây con | `O(n)` với hash map |

In [10]:
def luy_thua_nhanh(x, n, sau=0):
    if n == 0:
        return 1
    nua = luy_thua_nhanh(x, n // 2, sau + 1)
    kq = nua * nua * (x if n % 2 else 1)
    print("    " * sau + f"{x}^{n} = ({x}^{n // 2})²{' · ' + str(x) if n % 2 else ''} = {kq}")
    return kq

_ = luy_thua_nhanh(2, 10)

            2^1 = (2^0)² · 2 = 2
        2^2 = (2^1)² = 4
    2^5 = (2^2)² · 2 = 32
2^10 = (2^5)² = 1024


Lỗi hay gặp: chia không đều hoặc không nhỏ đi (`mid` tính sai → đệ quy vô hạn); tạo bản sao `a[:mid]` ở mỗi tầng (tốn thêm `O(n log n)` bộ nhớ — truyền chỉ số `lo, hi` thay vì cắt list); quên trường hợp đoạn rỗng.

## 2.4 Áp dụng vào bài này

**Merge sort** là lựa chọn tự nhiên cho **danh sách liên kết**: **chia** đôi bằng nhanh/chậm, sắp xếp hai nửa (**đệ quy**), **trộn** hai danh sách đã sắp xếp chỉ bằng đổi con trỏ. Bản bottom-up trộn các khối `1, 2, 4, …` để đạt `O(1)` bộ nhớ.

# 3. Nền tảng

In [11]:
from typing import List, Optional
import random, time, sys
sys.setrecursionlimit(10000)


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head, limit=10**6):
    out = []
    while head and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out

## Trộn hai danh sách đã sắp xếp (#21)

Viên gạch quan trọng nhất: nối lần lượt nút nhỏ hơn vào sau `tail`, dùng nút giả. **Không tạo nút mới**, `O(1)` bộ nhớ.

In [12]:
def merge(a: Optional[ListNode], b: Optional[ListNode]) -> Optional[ListNode]:
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:                 # <= giữ ổn định
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a or b
    return dummy.next

print(to_list(merge(build_list([1, 4, 7]), build_list([2, 3, 8, 9]))))

[1, 2, 3, 4, 7, 8, 9]


# 4. Baseline: chép ra mảng, sắp xếp, ghi lại

Dễ nhất — nhưng `O(n)` bộ nhớ, và thường không phải điều người phỏng vấn muốn.

In [13]:
def sort_via_array(head: Optional[ListNode]) -> Optional[ListNode]:
    nodes = []
    while head:
        nodes.append(head)
        head = head.next
    nodes.sort(key=lambda n: n.val)        # sort của Python ổn định
    for a, b in zip(nodes, nodes[1:]):
        a.next = b
    if nodes:
        nodes[-1].next = None
    return nodes[0] if nodes else None

print(to_list(sort_via_array(build_list([4, 2, 1, 3]))))

[1, 2, 3, 4]


# 5. Ý tưởng chính 1: merge sort top-down

```text
sort(head):
    if head is None or head.next is None: return head
    tìm giữa: slow dừng ở CUỐI nửa đầu
    second = slow.next; slow.next = None      # cắt đôi
    return merge(sort(head), sort(second))
```

Tìm giữa với `fast = head.next` (hoặc điều kiện `fast.next and fast.next.next`) để `slow` dừng ở **cuối nửa đầu** — với 2 nút, cắt thành `1 + 1`, không bị đệ quy vô hạn.

In [14]:
def sort_top_down(head: Optional[ListNode]) -> Optional[ListNode]:
    if not head or not head.next:
        return head
    slow, fast = head, head.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    second = slow.next
    slow.next = None
    return merge(sort_top_down(head), sort_top_down(second))

print(to_list(sort_top_down(build_list([4, 2, 1, 3]))), to_list(sort_top_down(build_list([-1, 5, 3, 4, 0]))))

[1, 2, 3, 4] [-1, 0, 3, 4, 5]


# 6. Mô phỏng top-down

In [15]:
def trace_top_down(values):
    def sort(head, depth):
        ind = "  " * depth
        if not head or not head.next:
            return head
        slow, fast = head, head.next
        while fast and fast.next:
            slow = slow.next; fast = fast.next.next
        second = slow.next; slow.next = None
        print(f"{ind}chia {to_list(head)} | {to_list(second)}")
        a = sort(head, depth + 1)
        b = sort(second, depth + 1)
        la, lb = to_list(a), to_list(b)          # ghi lại TRƯỚC khi merge đổi con trỏ
        m = merge(a, b)
        print(f"{ind}trộn {la} + {lb} = {to_list(m)}")
        return m
    return to_list(sort(build_list(values), 0))

_ = trace_top_down([-1, 5, 3, 4, 0])

chia [-1, 5, 3] | [4, 0]
  chia [-1, 5] | [3]
    chia [-1] | [5]
    trộn [-1] + [5] = [-1, 5]
  trộn [-1, 5] + [3] = [-1, 3, 5]
  chia [4] | [0]
  trộn [4] + [0] = [0, 4]
trộn [-1, 3, 5] + [0, 4] = [-1, 0, 3, 4, 5]


# 7. Ý tưởng chính 2: merge sort bottom-up — `O(1)` bộ nhớ

Không đệ quy. Với `size = 1, 2, 4, …` (tới khi `size ≥ n`): đi dọc danh sách, mỗi lần **cắt ra hai đoạn** dài `size`, trộn chúng, rồi nối kết quả vào sau phần đã xử lý.

Hai hàm phụ:

- `split(head, size)`: cắt sau `size` nút, trả về **đầu phần còn lại**.
- `merge` như trên, cộng thêm việc trả về **đuôi** của đoạn đã trộn (để nối tiếp).

In [16]:
def split(head: Optional[ListNode], size: int) -> Optional[ListNode]:
    """Giữ size nút đầu của head, cắt và trả về phần còn lại."""
    for _ in range(size - 1):
        if not head:
            break
        head = head.next
    if not head:
        return None
    rest = head.next
    head.next = None
    return rest


def merge_with_tail(a, b, tail):
    """Trộn a, b và gắn sau tail; trả về đuôi mới."""
    while a and b:
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a or b
    while tail.next:
        tail = tail.next
    return tail


def sort_bottom_up(head: Optional[ListNode]) -> Optional[ListNode]:
    n, node = 0, head
    while node:
        n += 1
        node = node.next
    dummy = ListNode(0, head)
    size = 1
    while size < n:
        prev, cur = dummy, dummy.next
        while cur:
            left = cur
            right = split(left, size)
            cur = split(right, size)
            prev = merge_with_tail(left, right, prev)
        size *= 2
    return dummy.next

print(to_list(sort_bottom_up(build_list([4, 2, 1, 3]))), to_list(sort_bottom_up(build_list([-1, 5, 3, 4, 0]))))

[1, 2, 3, 4] [-1, 0, 3, 4, 5]


In [17]:
def trace_bottom_up(values):
    head = build_list(values)
    n = len(values)
    dummy = ListNode(0, head)
    size = 1
    while size < n:
        prev, cur = dummy, dummy.next
        while cur:
            left = cur
            right = split(left, size)
            cur = split(right, size)
            prev = merge_with_tail(left, right, prev)
        print(f"size = {size:>2}: {to_list(dummy.next)}")
        size *= 2
    return to_list(dummy.next)

_ = trace_bottom_up([8, 3, 6, 1, 7, 2, 5, 4, 0])

size =  1: [3, 8, 1, 6, 2, 7, 4, 5, 0]
size =  2: [1, 3, 6, 8, 2, 4, 5, 7, 0]
size =  4: [1, 2, 3, 4, 5, 6, 7, 8, 0]
size =  8: [0, 1, 2, 3, 4, 5, 6, 7, 8]


Sau lượt `size = 1`, mọi cặp liền kề đã sắp xếp; sau lượt `size = 2`, mọi khối 4 đã sắp xếp; …

# 8. Phiên bản đúng format LeetCode

Dùng bottom-up để đạt follow-up.

In [18]:
class Solution:
    def sortList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        n, node = 0, head
        while node:
            n += 1
            node = node.next
        dummy = ListNode(0, head)
        size = 1
        while size < n:
            prev, cur = dummy, dummy.next
            while cur:
                left = cur
                right = split(left, size)
                cur = split(right, size)
                prev = merge_with_tail(left, right, prev)
            size *= 2
        return dummy.next

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


def run(f, values):
    head = build_list(values)
    ids = set()
    n = head
    while n:
        ids.add(id(n)); n = n.next
    out = f(head)
    res, got, n = [], set(), out
    while n and len(res) <= len(values):
        res.append(n.val); got.add(id(n)); n = n.next
    assert len(res) == len(values) and got == ids, "dùng lại đúng các nút, không vòng"
    return res


sol = Solution()
check("Ví dụ 1", run(sol.sortList, [4, 2, 1, 3]), [1, 2, 3, 4])
check("Ví dụ 2", run(sol.sortList, [-1, 5, 3, 4, 0]), [-1, 0, 3, 4, 5])
check("Rỗng", run(sol.sortList, []), [])
check("Một nút", run(sol.sortList, [1]), [1])
check("Trùng", run(sol.sortList, [2, 1, 2, 1, 2]), [1, 1, 2, 2, 2])

random.seed(148)
for _ in range(3000):
    vals = [random.randint(-10, 10) for _ in range(random.randint(0, 25))]
    exp = sorted(vals)
    assert run(sol.sortList, vals) == exp == run(sort_top_down, vals) == run(sort_via_array, vals)
print("3000 danh sách ngẫu nhiên: 3 cách khớp sorted()")

big = [random.randint(-10**5, 10**5) for _ in range(5 * 10**4)]
for name, f in [("bottom-up", sol.sortList), ("top-down", sort_top_down)]:
    t0 = time.time(); r = run(f, big); t1 = time.time()
    print(f"5·10^4 nút, {name:<9}: {t1 - t0:.3f}s, đúng? {r == sorted(big)}")

Ví dụ 1          PASS  [1, 2, 3, 4]


Ví dụ 2          PASS  [-1, 0, 3, 4, 5]
Rỗng             PASS  []
Một nút          PASS  [1]
Trùng            PASS  [1, 1, 2, 2, 2]


3000 danh sách ngẫu nhiên: 3 cách khớp sorted()


5·10^4 nút, bottom-up: 0.624s, đúng? True


5·10^4 nút, top-down : 0.537s, đúng? True


# 9. Vì sao đúng? Độ phức tạp?

**Top-down:** quy nạp — hai nửa được sắp xếp đệ quy, `merge` hai danh sách đã sắp xếp cho danh sách đã sắp xếp.

**Bottom-up:** bất biến "trước lượt `size`, danh sách gồm các khối liên tiếp độ dài `size` (khối cuối có thể ngắn hơn), mỗi khối đã sắp xếp". Lượt `size` trộn từng cặp khối → khối độ dài `2·size` đã sắp xếp. Khi `size ≥ n` chỉ còn một khối.

```text
                 Time          Bộ nhớ phụ
Top-down         O(n log n)    O(log n)  (ngăn xếp đệ quy)
Bottom-up        O(n log n)    O(1)
Insertion (#147) O(n²)         O(1)
```

`merge` trên danh sách chỉ đổi con trỏ — không cần mảng phụ `O(n)` như merge sort trên mảng.

# 10. Những lỗi dễ mắc

## Lỗi 1: Tìm giữa sai → đệ quy vô hạn với 2 nút

In [20]:
def wrong_middle(head):
    if not head or not head.next:
        return head
    slow = fast = head
    while fast and fast.next:                      # SAI: với 2 nút, slow dừng ở nút THỨ HAI
        slow = slow.next
        fast = fast.next.next
    second = slow.next
    slow.next = None
    return merge(wrong_middle(head), wrong_middle(second))

try:
    wrong_middle(build_list([2, 1]))
except RecursionError:
    print("RecursionError: [2, 1] bị chia thành [2, 1] + [] mãi mãi")

RecursionError: [2, 1] bị chia thành [2, 1] + [] mãi mãi


Với 2 nút, `slow` dừng ở nút thứ hai, cắt sau nó → nửa đầu vẫn là cả danh sách. Phải để `slow` dừng ở **cuối nửa đầu** (khởi tạo `fast = head.next`).

## Lỗi 2: Quên cắt (`slow.next = None`)

Nửa đầu vẫn nối sang nửa sau → `sort(head)` xử lý **cả danh sách** lần nữa → đệ quy vô hạn.

## Lỗi 3: `merge` tạo nút mới

Kết quả đúng, nhưng tốn `O(n)` bộ nhớ ở mỗi tầng và không còn là "sắp xếp danh sách tại chỗ".

## Lỗi 4: Bottom-up quên nối đoạn đã trộn vào phần trước

In [21]:
def wrong_bottom_up(head):
    n, node = 0, head
    while node:
        n += 1; node = node.next
    dummy = ListNode(0, head)
    size = 1
    while size < n:
        cur = dummy.next
        prev = dummy
        while cur:
            left = cur
            right = split(left, size)
            cur = split(right, size)
            merge_with_tail(left, right, ListNode())   # SAI: gắn vào nút tạm, không gắn vào prev
        size *= 2
    return dummy.next

print("Sai :", to_list(wrong_bottom_up(build_list([4, 2, 1, 3])), 10), "| Đúng:", to_list(sol.sortList(build_list([4, 2, 1, 3]))))

Sai : [4] | Đúng: [1, 2, 3, 4]


Các đoạn được trộn vào một nút tạm rồi bị bỏ; `dummy.next` vẫn trỏ vào nút `4` ban đầu, nhưng `split` đã cắt `4.next = None` → chỉ còn `[4]`, **mất** 3 nút. Luôn gắn kết quả trộn sau `prev` rồi đặt `prev` = đuôi mới.

# 11. Mẹo nhận diện pattern

- **Sắp xếp danh sách liên kết** → merge sort (không cần truy cập ngẫu nhiên; trộn `O(1)` bộ nhớ).
- **Chia để trị trên danh sách** → tìm giữa bằng nhanh/chậm + cắt.
- **Muốn bỏ đệ quy** → bottom-up với kích thước tăng gấp đôi.

Bài liên quan: **#21 Merge Two Sorted Lists**, **#23 Merge k Sorted Lists** (Hard — trộn theo cặp hoặc heap), **#147 Insertion Sort List**, **#912 Sort an Array**, **#88 Merge Sorted Array**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chạy bottom-up cho `5 → 1 → 4 → 2 → 3` và ghi lại danh sách sau mỗi lượt `size`.

## Bài 2 — Điền chỗ trống (top-down)

In [22]:
def exercise_2_sort_list(head: Optional[ListNode]) -> Optional[ListNode]:
    if not head or not head.next:
        return head
    # TODO: tìm giữa (slow dừng ở cuối nửa đầu), cắt, đệ quy, trộn
    pass


# assert to_list(exercise_2_sort_list(build_list([4, 2, 1, 3]))) == [1, 2, 3, 4]
# assert to_list(exercise_2_sort_list(build_list([-1, 5, 3, 4, 0]))) == [-1, 0, 3, 4, 5]
# assert to_list(exercise_2_sort_list(build_list([]))) == []

## Bài 3 — Biến thể: Merge k Sorted Lists (LeetCode #23, Hard)

Trộn `k` danh sách đã sắp xếp. Hai cách: trộn theo cặp (giống bottom-up, `O(N log k)`) hoặc dùng heap.

```python
merge_k([[1, 4, 5], [1, 3, 4], [2, 6]]) -> [1, 1, 2, 3, 4, 4, 5, 6]
```

In [23]:
def exercise_3_merge_k(lists: List[Optional[ListNode]]) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_3_merge_k([build_list([1, 4, 5]), build_list([1, 3, 4]), build_list([2, 6])])) == [1, 1, 2, 3, 4, 4, 5, 6]
# assert exercise_3_merge_k([]) is None
# assert exercise_3_merge_k([None]) is None

## Bài 4 — Biến thể: quick sort trên danh sách liên kết

Dùng phân hoạch 3 nhóm (`<`, `==`, `>` pivot) như #86, đệ quy hai nhóm ngoài, rồi nối. (Trường hợp xấu `O(n²)`, chọn pivot ngẫu nhiên để giảm rủi ro.)

In [24]:
def exercise_4_quick_sort_list(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_4_quick_sort_list(build_list([4, 2, 1, 3]))) == [1, 2, 3, 4]
# assert to_list(exercise_4_quick_sort_list(build_list([3, 3, 1, 3]))) == [1, 3, 3, 3]
# assert to_list(exercise_4_quick_sort_list(build_list([]))) == []

## Bài 5 — Mini challenge: đếm nghịch thế bằng merge sort trên danh sách

Đếm số cặp `(i < j)` với `a[i] > a[j]` khi `a` cho dưới dạng danh sách liên kết, `O(n log n)`. Trong bước trộn, mỗi khi lấy nút từ nửa **phải**, cộng số nút **còn lại** ở nửa trái.

In [25]:
def exercise_5_count_inversions(head: Optional[ListNode]) -> int:
    # TODO
    pass


# assert exercise_5_count_inversions(build_list([4, 2, 1, 3])) == 4
# assert exercise_5_count_inversions(build_list([1, 2, 3])) == 0
# assert exercise_5_count_inversions(build_list([])) == 0

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_bottom_up([5, 1, 4, 2, 3])` sau khi tự làm.

**Bài 3**
- Mức 1 (theo cặp): `while len(lists) > 1`: trộn `lists[0]` với `lists[1]`, `lists[2]` với `lists[3]`, … thành danh sách mới.
- Mức 2 (heap): đẩy `(val, i, node)` của đầu mỗi danh sách; lấy nhỏ nhất, đẩy `node.next`.

**Bài 4**
- Mức 1: chọn pivot = giá trị của một nút ngẫu nhiên; tách 3 danh sách bằng 3 nút giả.

**Bài 5**
- Mức 1: hàm `sort(head)` trả `(head_đã_sắp_xếp, số_nghịch_thế, độ_dài)`.
- Mức 2: khi trộn, nếu lấy từ phải thì `inv += len_trái_còn_lại`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
_ = trace_bottom_up([5, 1, 4, 2, 3])

size =  1: [1, 5, 2, 4, 3]
size =  2: [1, 2, 4, 5, 3]
size =  4: [1, 2, 3, 4, 5]


## Đáp án Bài 2

In [27]:
def answer_2_sort_list(head: Optional[ListNode]) -> Optional[ListNode]:
    return sort_top_down(head)

assert to_list(answer_2_sort_list(build_list([4, 2, 1, 3]))) == [1, 2, 3, 4]
assert to_list(answer_2_sort_list(build_list([-1, 5, 3, 4, 0]))) == [-1, 0, 3, 4, 5]
assert to_list(answer_2_sort_list(build_list([]))) == []
print("Bài 2 PASS (giống sort_top_down ở mục 5)")

Bài 2 PASS (giống sort_top_down ở mục 5)


## Đáp án Bài 3

In [28]:
import heapq

def answer_3_merge_k(lists: List[Optional[ListNode]]) -> Optional[ListNode]:
    lists = [l for l in lists if l]
    if not lists:
        return None
    while len(lists) > 1:
        merged = []
        for i in range(0, len(lists), 2):
            merged.append(merge(lists[i], lists[i + 1]) if i + 1 < len(lists) else lists[i])
        lists = merged
    return lists[0]

def merge_k_heap(lists):
    heap = [(l.val, i, l) for i, l in enumerate(lists) if l]
    heapq.heapify(heap)
    dummy = tail = ListNode()
    while heap:
        _, i, node = heapq.heappop(heap)
        tail.next = node; tail = node
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

assert to_list(answer_3_merge_k([build_list([1, 4, 5]), build_list([1, 3, 4]), build_list([2, 6])])) == [1, 1, 2, 3, 4, 4, 5, 6]
assert answer_3_merge_k([]) is None
assert answer_3_merge_k([None]) is None
for _ in range(1000):
    arrs = [sorted(random.randint(0, 20) for _ in range(random.randint(0, 6))) for _ in range(random.randint(0, 6))]
    exp = sorted(x for a in arrs for x in a)
    assert to_list(answer_3_merge_k([build_list(a) for a in arrs])) == exp == to_list(merge_k_heap([build_list(a) for a in arrs]))
print("Bài 3 PASS (theo cặp và heap)")

Bài 3 PASS (theo cặp và heap)


## Đáp án Bài 4

In [29]:
def answer_4_quick_sort_list(head: Optional[ListNode]) -> Optional[ListNode]:
    if not head or not head.next:
        return head
    vals, n = [], head
    while n:
        vals.append(n.val); n = n.next
    pivot = random.choice(vals)
    lt = lt_t = ListNode(); eq = eq_t = ListNode(); gt = gt_t = ListNode()
    n = head
    while n:
        nxt = n.next
        n.next = None
        if n.val < pivot:
            lt_t.next = n; lt_t = n
        elif n.val == pivot:
            eq_t.next = n; eq_t = n
        else:
            gt_t.next = n; gt_t = n
        n = nxt
    left = answer_4_quick_sort_list(lt.next)
    right = answer_4_quick_sort_list(gt.next)
    eq_t.next = right
    if not left:
        return eq.next
    t = left
    while t.next:
        t = t.next
    t.next = eq.next
    return left

assert to_list(answer_4_quick_sort_list(build_list([4, 2, 1, 3]))) == [1, 2, 3, 4]
assert to_list(answer_4_quick_sort_list(build_list([3, 3, 1, 3]))) == [1, 3, 3, 3]
assert to_list(answer_4_quick_sort_list(build_list([]))) == []
for _ in range(1000):
    v = [random.randint(-5, 5) for _ in range(random.randint(0, 20))]
    assert to_list(answer_4_quick_sort_list(build_list(v))) == sorted(v)
print("Bài 4 PASS")

Bài 4 PASS


(Để chọn pivot ngẫu nhiên ta đã đọc giá trị ra một list tạm — đơn giản cho bài tập; có thể chọn bằng reservoir sampling để giữ `O(1)` bộ nhớ.)

## Đáp án Bài 5

In [30]:
def answer_5_count_inversions(head: Optional[ListNode]) -> int:
    def sort(h, n):
        if n <= 1:
            if h:
                h.next = None
            return h, 0
        mid = n // 2
        second = h
        for _ in range(mid):
            second = second.next
        a, x = sort(h, mid)                 # sort(h, mid) cắt sau mid nút
        b, y = sort(second, n - mid)
        inv = x + y
        dummy = tail = ListNode()
        left_remaining = mid
        while a and b:
            if a.val <= b.val:
                tail.next, a = a, a.next
                left_remaining -= 1
            else:
                tail.next, b = b, b.next
                inv += left_remaining
            tail = tail.next
        tail.next = a or b
        return dummy.next, inv
    n, node = 0, head
    while node:
        n += 1; node = node.next
    return sort(head, n)[1]

assert answer_5_count_inversions(build_list([4, 2, 1, 3])) == 4
assert answer_5_count_inversions(build_list([1, 2, 3])) == 0
assert answer_5_count_inversions(build_list([])) == 0
for _ in range(2000):
    a = [random.randint(0, 6) for _ in range(random.randint(0, 12))]
    assert answer_5_count_inversions(build_list(a)) == sum(a[i] > a[j] for i in range(len(a)) for j in range(i + 1, len(a)))
print("Bài 5 PASS")

Bài 5 PASS

Chú ý trong `sort(h, n)`: ta tìm `second` **trước** khi đệ quy vào nửa trái (vì đệ quy sẽ cắt và sắp xếp lại các nút của nửa trái).

# 15. Tổng kết

> Merge sort cho danh sách liên kết: top-down (nhanh/chậm tìm cuối nửa đầu, cắt, đệ quy, trộn) hoặc bottom-up (trộn khối `1, 2, 4, …` bằng `split` + `merge`, `O(1)` bộ nhớ). `O(n log n)`, ổn định nhờ `<=` khi trộn.

---

## Checklist trước khi nộp

- [ ] Tìm giữa để `slow` ở **cuối nửa đầu** (`fast = head.next`)?
- [ ] Cắt `slow.next = None`?
- [ ] `merge` chỉ đổi con trỏ, dùng `<=`?
- [ ] Bottom-up: nối đoạn đã trộn vào `prev` và cập nhật `prev` thành đuôi mới?